# Indexing Prior Sessions and Retrieving a Past Root Cause

In this demo, we turn past CI triage sessions into memory records with provenance, store them in Chroma, and find the past root cause of a new failure. Then we check out a newer version of the code, and see that the remembered workaround no longer holds.

In [1]:
import sys
from pathlib import Path

from dotenv import load_dotenv

load_dotenv(".env")

sys.path.insert(0, str(Path("..").resolve()))
from memory_store import DATA, REPO, STORE, load_memories, search

MODEL = "gpt-5.2"
SESSION = DATA / "sessions" / "s5.md"

In [2]:
import subprocess


def git(*args: str) -> str:
    """Run a git command in the project repo and return its output."""
    return subprocess.run(
        ["git", *args], cwd=REPO, capture_output=True, text=True, check=True
    ).stdout.strip()


# Start from the old tag, the code the sessions were written against.
git("checkout", "-q", "old")
print(f"invoicekit at the old tag: {git('rev-parse', '--short', 'HEAD')}")

invoicekit at the old tag: 78bdd66


## 1. A Triage Session

Notes from a past session: each finding has the failure, its root cause, the workaround, the files involved, and the line the workaround depends on.

In [3]:
print(SESSION.read_text())

# Triage session s5

Date: 2026-03-03
Repo: invoicekit
Commit: 78bdd666849a0eff48efed79c39d465244cd0741

## Finding m09

We looked at a failing build: The Python 3.9 job fails: TypeError: unsupported operand type(s) for |: 'type' and 'NoneType'.
Root cause: invoice.py uses `str | None`, which Python 3.9 can't evaluate.
Workaround: Ignore it: the 3.9 job is allowed to fail with continue-on-error in the CI workflow.
Files we looked at: .github/workflows/ci.yml, src/invoicekit/invoice.py.
The line this depends on, in .github/workflows/ci.yml: `continue-on-error: ${{ matrix.python-version == '3.9' }}`

## Finding m10

We looked at a failing build: test_today_is_a_date fails around midnight: assert datetime.date(2026, 3, 2) <= datetime.date(2026, 3, 1).
Root cause: today() uses the machine's local time, which differs from the date the test expects.
Workaround: CI sets TZ: UTC in the workflow env. Set TZ=UTC locally too.
Files we looked at: .github/workflows/ci.yml, src/invoicekit/dates.py.


## 2. Extract Memory Records

We ask the model to turn each finding into a memory record, with its provenance: repo, commit SHA, files and session.

In [4]:
from openai import OpenAI
from pydantic import BaseModel


class MemoryRecord(BaseModel):
    id: str
    repo: str
    commit: str
    session: str
    files: list[str]
    failure: str
    root_cause: str
    workaround: str
    evidence: str


class Session(BaseModel):
    memories: list[MemoryRecord]


response = OpenAI().responses.parse(
    model=MODEL,
    input=(
        "Turn each finding in this triage session into a memory record. Take the repo, commit "
        "and session from the header, and the evidence from the line the finding depends on.\n\n"
        + SESSION.read_text()
    ),
    text_format=Session,
)
for record in response.output_parsed.memories:
    print(
        f"{record.id}  session={record.session}  repo={record.repo}  commit={record.commit[:12]}"
    )
    print(f"  files:      {', '.join(record.files)}")
    print(f"  root cause: {record.root_cause}")
    print(f"  workaround: {record.workaround}")
    print(f"  evidence:   {record.evidence}\n")

m09  session=s5  repo=invoicekit  commit=78bdd666849a
  files:      .github/workflows/ci.yml, src/invoicekit/invoice.py
  root cause: invoice.py uses the PEP 604 union type syntax (`str | None`), which Python 3.9 cannot evaluate.
  workaround: Ignore the failure: the Python 3.9 job is configured as allowed-to-fail via continue-on-error in the CI workflow.
  evidence:   `continue-on-error: ${{ matrix.python-version == '3.9' }}`

m10  session=s5  repo=invoicekit  commit=78bdd666849a
  files:      .github/workflows/ci.yml, src/invoicekit/dates.py
  root cause: today() uses the machine's local time, which can differ from the date the test expects depending on timezone and the time the test runs.
  workaround: Use UTC timezone to stabilize the date behavior: CI sets TZ=UTC; set TZ=UTC locally as well.
  evidence:   `TZ: UTC`

m11  session=s5  repo=invoicekit  commit=78bdd666849a
  files:      src/invoicekit/tax.py, tests/test_tax.py
  root cause: vat rounding uses half-to-even (bankers' rou

## 3. Store the Memories

`data/memories.jsonl` holds the records from all five sessions. We store each one in Chroma, a vector database:

- its text (failure, root cause and workaround) as an embedding, so we can search it by meaning;
- its details and provenance as metadata: repo, commit SHA, files, session, the evidence line, when it was last used, and its status.

In [5]:
import json
import os

import chromadb
from chromadb.utils.embedding_functions.openai_embedding_function import (
    OpenAIEmbeddingFunction,
)

client = chromadb.PersistentClient(path=str(STORE))

embed = OpenAIEmbeddingFunction(
    api_key=os.environ["OPENAI_API_KEY"],
    model_name="text-embedding-3-small",
    api_base=os.environ.get("OPENAI_BASE_URL"),
)

collection = client.get_or_create_collection(
    name="memories",
    embedding_function=embed,
    configuration={"hnsw": {"space": "cosine"}},
)

memories = load_memories()
# upsert adds each memory, or replaces it if it's already there from an earlier run.
collection.upsert(
    ids=[m["id"] for m in memories],
    # The text we search by meaning: what failed, why, and how to fix it.
    documents=[
        f"{m['failure']}\nRoot cause: {m['root_cause']}\nWorkaround: {m['workaround']}"
        for m in memories
    ],
    metadatas=[
        {
            "repo": m["repo"],
            "commit_sha": m["commit"],
            "session": m["session"],
            "files": json.dumps(m["files"]),
            "evidence_file": m["evidence_file"],
            "evidence": m["evidence"],
            "failure": m["failure"],
            "root_cause": m["root_cause"],
            "workaround": m["workaround"],
            "last_used_session": m["last_used_session"],
            "status": "unverified",
            "status_reason": "",
        }
        for m in memories
    ],
)
print(f"{collection.count()} memories in Chroma")

12 memories in Chroma


Here is memory m10, as Chroma stores it.

In [6]:
stored = collection.get(ids=["m10"], include=["documents", "metadatas", "embeddings"])
print("Content:")
print("  " + stored["documents"][0].replace("\n", "\n  "))

metadata = stored["metadatas"][0]
print("\nMetadata:")
for key in (
    "repo",
    "session",
    "commit_sha",
    "files",
    "evidence_file",
    "evidence",
    "last_used_session",
    "status",
):
    print(f"  {key:<18} {metadata[key]}")

embedding = stored["embeddings"][0]
print(
    f"\nIts embedding: {len(embedding)} numbers, starting {[round(float(x), 3) for x in embedding[:4]]}"
)

Content:
  test_today_is_a_date fails around midnight: assert datetime.date(2026, 3, 2) <= datetime.date(2026, 3, 1)
  Root cause: today() uses the machine's local time, which differs from the date the test expects.
  Workaround: CI sets TZ: UTC in the workflow env. Set TZ=UTC locally too.

Metadata:
  repo               invoicekit
  session            s5
  commit_sha         78bdd666849a0eff48efed79c39d465244cd0741
  files              [".github/workflows/ci.yml", "src/invoicekit/dates.py"]
  evidence_file      .github/workflows/ci.yml
  evidence           TZ: UTC
  last_used_session  4
  status             unverified

Its embedding: 1536 numbers, starting [-0.013, -0.02, 0.107, -0.053]


## 4. Recall a Past Root Cause

A new build fails. We search the memories by meaning, in the same repo only.

In [7]:
NEW_FAILURE = (
    "FAILED tests/test_dates.py::test_today_is_a_date - "
    "assert datetime.date(2026, 4, 2) <= datetime.date(2026, 4, 1)"
)

print(f"New failure:\n  {NEW_FAILURE}\n\nClosest memories:\n")
for memory, distance in search(NEW_FAILURE, repo="invoicekit"):
    print(
        f"  {distance:.3f}  {memory['id']}  from session {memory['session']}, commit {memory['commit_sha'][:12]}"
    )
    print(f"         root cause: {memory['root_cause']}")
    print(f"         workaround: {memory['workaround']}\n")

New failure:
  FAILED tests/test_dates.py::test_today_is_a_date - assert datetime.date(2026, 4, 2) <= datetime.date(2026, 4, 1)

Closest memories:

  0.289  m10  from session s5, commit 78bdd666849a
         root cause: today() uses the machine's local time, which differs from the date the test expects.
         workaround: CI sets TZ: UTC in the workflow env. Set TZ=UTC locally too.

  0.433  m08  from session s4, commit 78bdd666849a
         root cause: due_date moves a due date that falls on a weekend to the next Monday. A local patch removed that loop.
         workaround: Keep the weekday loop in due_date.

  0.655  m01  from session s1, commit 78bdd666849a
         root cause: A test built line amounts from floats, so the total picked up float error.
         workaround: Build amounts from strings, e.g. Decimal("0.10"). Invoice.total already sums Decimals.



## 5. The Code Moved On

We check out the new tag. The closest memory's workaround depends on one line of the code: `git log -S` (the pickaxe) finds the commit that added or removed that line since the memory was written.

In [10]:
memory, _ = search(NEW_FAILURE, repo="invoicekit")[0]
git("checkout", "-q", "new")
print(f"Checked out the new tag: {git('rev-parse', '--short', 'HEAD')}")
print(f"The memory {memory['id']} says: {memory['workaround']}")
print(f"It depends on this line in {memory['evidence_file']}: {memory['evidence']!r}\n")

print("The commit that removed it (git log -S, the pickaxe):")
print(
    "  "
    + git(
        "log",
        "-S",
        memory["evidence"],
        "--format=%h %s",
        f"{memory['commit_sha']}..HEAD",
        "--",
        memory["evidence_file"],
    )
)

Checked out the new tag: 8bb3031
The memory m10 says: CI sets TZ: UTC in the workflow env. Set TZ=UTC locally too.
It depends on this line in .github/workflows/ci.yml: 'TZ: UTC'

The commit that removed it (git log -S, the pickaxe):
  f828f54 Use timezone-aware dates


In [11]:
# Put the project back on the old tag.
git("checkout", "-q", "old")

''